In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Literal
from pydantic import BaseModel, Field

In [ ]:
load_dotenv()

In [ ]:
model = ChatOpenAI(model ='gpt-4o-mini')

In [ ]:
class SentimentSchema(BaseModel):
    
    sentiment: Literal["positive", "negative"] = Field(description='Sentiment of the review')

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user in the review')
    urgency: Literal["Low", "Medium", "High"] = Field(description='How urgent or critical the issue appears to be')

In [ ]:
structured_model0 = model.with_structured_output(SentimentSchema)
structured_model1 = model.with_structured_output(DiagnosisSchema)

In [ ]:
class ReviewState(TypedDict):

    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [ ]:
def find_sentiment(state: ReviewState):

    prompt = f'For the following review find out the sentiment \n {state["review"]}'

    sentiment = structured_model0.invoke(prompt)

    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:

    if state['sentiment'] == "positive":
        return 'positive_response'
    else:
        return "run_diagnosis"
    
def positive_response(state: ReviewState):

    prompt: f"""Write a warm thank-you message in reponse to this review:
    \n\n\"{state['review']}\"\n
    Also, kindly ask the user to leave feedback on our website.""" 

    response = model.invoke(prompt)
    return {'response': response}

def run_diagnosis(state: ReviewState):

    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n
    Return issue_type, tone, and urgency."""

    response = structured_model1.invoke(prompt)

    return {'diagnosis': response.model_dump()}

def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    Write an empathetic, helpful resolution message.
    """
    response = model.invoke(prompt).content

    return {'response': response}

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('find_sentiment', END)

workflow = graph.complie()

In [ ]:
initial_state = {
    'review': 'The product is great'
}

workflow.invoke(initial_state)